In [ ]:
''' 
Plot individual boards from action prediction experiment
'''

import ast
import os
import sys
sys.path.append(os.path.abspath('..'))

import models.utils as utils 
import models.agents.uni_random as uni_random
import models.agents.heuristics as heuristics
import models.agents.mcts as mcts
import analysis.analysis_utils as analysis_utils
import matplotlib
import matplotlib.pylab as plt
from matplotlib.patches import Circle, Rectangle
import seaborn as sns 
import itertools 
import pandas as pd
from tqdm import tqdm
from tqdm import trange
import random
import multiprocessing
import models.just_think.run_models 

import datetime
import importlib 
from itertools import chain
import constants
import numpy as np
import json

%matplotlib inline


save_dir = "play_figs/"

In [ ]:
"""
Load in human data
"""
with open(f"../human-data/play-exp/human-v-human/final-play/final_agg.json", "r") as f: 
    human_gameplay_data = json.load(f)

In [ ]:
import pickle

model_fit_data = {}

with open("../model-data/play-exp/final_data/heuristics.txt", "r") as f: 
    model_fit_data['ours'] = eval(f.readlines()[0])

# Update July 3 2025
expert_pickle_file_path = "../model-data/play-exp/2025-07-02_heuristic_search_eg.pickle"

with open(expert_pickle_file_path, 'rb') as file:
    loaded_data = pickle.load(file)
model_fit_data['expert'] = loaded_data

depth3_pickle_file_path = '../model-data/play-exp/final_data/depth3-new-may24.pickle'
with open(depth3_pickle_file_path, 'rb') as file:
    loaded_data = pickle.load(file)
model_fit_data['ours-depth3'] = loaded_data

In [ ]:
importlib.reload(models.just_think.run_models)
importlib.reload(analysis_utils)
from models.agents.uni_random import uniform_dist
considered_games = sorted(list(human_gameplay_data))
all_participant_data = {stim: {} for stim in considered_games}
all_agents = ['ours', 'ours-depth3', 'expert', 'random']
all_model_data = {agent: {stim: {} for stim in considered_games} for agent in all_agents}

games, idx2game, game2idx, game_stimuli = analysis_utils.process_game_stimuli(constants.THINK_STIMULI_PTH)

# map the game, arena, and move symbol to a particular player
participant_id_map = {}
participant2arena = {}

for agent in all_agents: 
    for idx, (stim, stim_data) in  enumerate(human_gameplay_data.items()): 
        game_id = game2idx[stim]
        game = games[game_id - 1]
        all_game_boards = [eval(board) for board in stim_data["boards"]]
        
        n_cols = len(all_game_boards[0])
        n_rows = len(all_game_boards[0][0])
        n_cells = n_cols * n_rows
        
        for arena_idx, game_boards in enumerate(all_game_boards): 
            arena_name = stim_data["arena"][arena_idx]
            
            order2player = stim_data['order2player'][arena_idx]
            player2order = {player: 'X' if order == '1' else 'O' for order, player in order2player.items()}
            
            for player, order in player2order.items(): 
                if player not in participant_id_map: 
                    participant_id_map[player] = {stim: order, "arena": arena_name}
                else: participant_id_map[player][stim] = order

            
            if len(game_boards) == 0: continue
            starting_board = analysis_utils.convert_board_repr(game_boards[0])
            prev_board = starting_board
                    
            arena_data_h = {}
            arena_data_m = {}
                    
            for i, current_board in enumerate(game_boards[1:]): 

                if agent == "random":
                    move_dist = [uniform_dist(prev_board)]
                else: 
                    try:
                        output = model_fit_data[agent][arena_name][str(game['index'])][str(i)]
                        # just one dist for now 
                        move_dist = output['dist']
                        if move_dist is None: 
                            print("No move dist found")
                    
                        if type(move_dist) != list:
                            move_dist = [move_dist] 
                        
                        m_outputs = []
                        for m_output in move_dist: 
                            new_dict = {}
                            for k, v in m_output.items():
                                if v != -np.inf: new_dict[k] = v
                            m_outputs.append(new_dict)
                        
                        move_dist = m_outputs
                        
                        
                    except Exception as e:
                        print("error: ", e, agent, stim, arena_name, str(i+1), str(game['index']), model_fit_data[agent][arena_name].keys()) 
                        continue 
                
                
                round_board = analysis_utils.convert_board_repr(current_board)

                # what was this player's next move?
                diff = np.array(round_board) != np.array(prev_board)  # Find where the boards differ
                row, col = np.where(diff)  # Get the indices of the differences

                # just one move
                move = (int(row[0]), int(col[0]))
                player_id = round_board[move[0]][move[1]]
                num_legal = np.sum(np.array(prev_board) == '_')
                
                arena_data_h[i] = (move, player_id)
                arena_data_m[i] = (move_dist, num_legal)                              
                prev_board = round_board             
                
            all_participant_data[stim][arena_name] = arena_data_h
            all_model_data[agent][stim][arena_name] = arena_data_m

In [ ]:
importlib.reload(analysis_utils)

viz_agents = ['ours', 'expert']
AGENT_TO_TITLE = {
    'human-play': 'Played',
    'should': 'Human',
    'ours': 'Intuitive',
    'expert': 'Expert',
    'random': 'Random'
}

In [ ]:
PLAYER_LIST = ["empty", "player_1", "player_2"]
COLORS = {
    # White / Black / Red (or purple) / Grey
    "player_1": "#ffffff", # white
    "player_2": "#000000", # black
    "target": "#8e10db", # purple
    "empty": "#c4c3c3"
}

# Use a colormap to interpolate with empty, and then take a moderate value of that colormap to use
# as the new baseline color
p1_cmap = matplotlib.colors.LinearSegmentedColormap.from_list("", [COLORS["empty"], COLORS["player_1"]])
COLORS["player_1_base"] = p1_cmap(0.1)

p2_cmap = matplotlib.colors.LinearSegmentedColormap.from_list("", [COLORS["empty"], COLORS["player_2"]])
COLORS["player_2_base"] = p2_cmap(0.1)

def dist_from_dict(dist_dict, height, width):
    dist = np.zeros((height, width))
    for key, val in dist_dict.items():
        dist[key] = val
    return dist

def render_dist(ax, distribution, board, cmap=None, played_move=None, player_to_move=None,
                scale_highlight=False):
    '''
    Render the specified distribution onto the provided figure axis. The provided
    board of existing moves will be drawn on top (to use a distribution of all zeros
    to just plot the board)
    '''

    height, width = distribution.shape

    if scale_highlight:
        # Want to scale the highlight radius down to ~0.15 when there are 100 squares
        highlight_rad = min(0.33, 0.33 * (50 / (height * width)))
    else:
        highlight_rad = 0.33

    # im = ax.imshow(distribution, cmap=cmap, vmin=0, vmax=1)
    im = ax.imshow(np.zeros_like(distribution), cmap=cmap, vmin=0, vmax=1)


    # Remove the ticks, separate the squares
    for x in np.arange(-0.5, width, 1):
        ax.axvline(x, color='w', linestyle='-', linewidth=1)
    for y in np.arange(-0.5, height, 1):
        ax.axhline(y, color='w', linestyle='-', linewidth=1)
    
    plt.setp(ax.spines.values(), visible=False)
    ax.tick_params(left=False, labelleft=False, bottom=False, labelbottom=False)

    for y in range(len(board)):
        for x in range(len(board[0])):
            val = board[y][x]
            player = PLAYER_LIST[val]

            if player != "empty":

                rect = Rectangle((x-0.5, y-0.5), 1, 1, color=COLORS["empty"])
                ax.add_patch(rect)
                
                circle = Circle((x, y), 0.333, color=COLORS[player])
                ax.add_patch(circle)

            else:
                dist_val = distribution[y][x]
                color = cmap(dist_val)

                rect = Rectangle((x-0.5, y-0.5), 1, 1, color=color)
                ax.add_patch(rect)

                if (y, x) == played_move:
                    highlight_circle = Circle((x, y), highlight_rad, color=COLORS[player_to_move], fill=False)
                    ax.add_patch(highlight_circle)
    
    return im


def plot_model_comparison(game, arena, board_idx, viz_agents, fig=None, game_index=None, title=None,
                          show_colorbar=True, colorbar_anchor=(0, 0.5), colorbar_location="right",
                          colorbar_pad=0.05):
    # First, obtain the index of the arena w.r.t. all of the arenas recorded for the given game
    arena_index = human_gameplay_data[game]['arena'].index(arena)

    # Then, parse out the list of boards for the specified game / arena into an actual list of lists
    boards_list = ast.literal_eval(human_gameplay_data[game]['boards'][arena_index])

    # Finally, obtain the actual board for the current and next move
    current_board = np.array(boards_list[board_idx])
    next_board = np.array(boards_list[board_idx+1])

    height, width = current_board.shape

    player_difference = (next_board - current_board).max().item()
    played_move = tuple([i.item() for i in np.unravel_index((next_board - current_board).argmax(), current_board.shape)])
    to_move = "player_1" if (player_difference == 1) else "player_2"

    if fig is None:
        fig, axes = plt.subplots(len(viz_agents), 1, figsize=(16, 6))
    else:
        axes = fig.subplots(len(viz_agents), 1)

    cmap = matplotlib.colors.LinearSegmentedColormap.from_list("", [COLORS["empty"], COLORS["target"]])

    # Plot the first agent
    agent = viz_agents[0]
    model_results = all_model_data[agent][game][arena]
    model_dist_dicts = model_results[board_idx][0]
    aggregated_dist_dict = analysis_utils.average_dicts([analysis_utils.softmax_dist(dist_dict, T=1) for dist_dict in model_dist_dicts])
    dist = dist_from_dict(aggregated_dist_dict, height, width)
    im = render_dist(axes[0], distribution=dist, board=current_board, cmap=cmap, played_move=played_move, player_to_move=to_move)
    axes[0].set_ylabel(AGENT_TO_TITLE[agent], fontweight="bold")

    if title is not None:
        formatted_title = title
    elif game_index is None:
        tidy_name = analysis_utils.tidy_game_code(game)
        fmt_name = "$\\bf{Game:}$ " + tidy_name
        fmt_arena = "$\\bf{Arena:}$ " + arena
        fmt_board = "$\\bf{Board:}$ " + str(board_idx)
        formatted_title = f"{fmt_name}\n{fmt_arena}\n{fmt_board}"
    else:
        formatted_title = f"Game Index: {game_index}\nArena: {arena}\nBoard #: {board_idx}"
    axes[0].set_title(formatted_title, fontsize=8)

    # Plot each of the agents using the *current* board
    for j, agent in enumerate(viz_agents[1:]): 
        model_results = all_model_data[agent][game][arena]
        model_dist_dicts = model_results[board_idx][0]
        aggregated_dist_dict = analysis_utils.average_dicts([analysis_utils.softmax_dist(dist_dict, T=1) for dist_dict in model_dist_dicts])
        dist = dist_from_dict(aggregated_dist_dict, height, width)
        render_dist(axes[j+1], distribution=dist, board=current_board, cmap=cmap, played_move=played_move, player_to_move=to_move)
        axes[j+1].set_ylabel(AGENT_TO_TITLE[agent], fontweight="bold")

    if show_colorbar:
        cbar = fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.75, anchor=colorbar_anchor,
                            location=colorbar_location, pad=colorbar_pad)
        # cbar.ax.tick_params(size=0)

    if fig is None:
        plt.show()

    return fig, axes

In [ ]:
''' 
Load and process human watch data
'''

with open("../human-data/watch-exp/final_res.json", "r") as f: 
    watch_data = json.load(f)

# rem IDs that used standard values too much [as pre-reg]
rem_ids = ['MUJ2ELF5D6', 'XAOHNXHKQL', 'FRF6G0BXYN', 'J090IDCESB', 'ZMSROEENK1', 'ZRSSZKF1VU', 'B8RXDIXOCL', 'M9SW86IBCL', '2WKOW6RBSH', 'O5CGUJIT5A']

In [ ]:
GAME_STAGES = ['early', 'intermediate', 'late']
order2stage = {i: v for i, v in enumerate(GAME_STAGES)}

move_idx_offset = -1

PLAYED_GAME_DATA = {}

for uid, uid_data in watch_data.items(): 
    if uid in rem_ids: continue
    for game, game_res in uid_data.items(): 
        batch_metadata = game_res['batch_metadata']
        arena = batch_metadata['arena'] 
        
        played_moves = batch_metadata['played_pos']
        
        if game not in PLAYED_GAME_DATA: PLAYED_GAME_DATA[game] = {}
        if arena not in PLAYED_GAME_DATA[game]: PLAYED_GAME_DATA[game][arena] = {}
        
        # get moves 'so far' for each board 
        boards = batch_metadata['boards']
        processed_boards = analysis_utils.process_game_states(boards)
        pred_move_idxs = batch_metadata['pred_move_idxs']
        
        # get game outcome 
        play_data = human_gameplay_data[game]
        player2order = None
        # make sure to pull out the "right" arena
        for arena2, order2player, outcome, all_move_times in zip(play_data['arena'],  
                                                 play_data["order2player"], 
                                                 play_data["outcome"],
                                                 play_data['move_times']): 
            if arena != arena2: continue
            player2order = {v: k for k,v in order2player.items()}

            if outcome == "Draw": 
                obs_outcome = 0 
            else: 
                obs_outcome = int(player2order[outcome])
                
            played_move_times = [all_move_times[idx] for idx in pred_move_idxs]
                
            break
        
        boards_at_stage = {}
        
        for order, move_idx in enumerate(pred_move_idxs): 
            stage = GAME_STAGES[order]
            boards_at_stage[stage] = processed_boards[move_idx + move_idx_offset]
            
        PLAYED_GAME_DATA[game][arena] = {
            'boards_at_stage': boards_at_stage,
            'played_pos': played_moves,
            'obs_outcome': obs_outcome,
            'move_idxs': pred_move_idxs, 
            'player2order': player2order, 
            'played_move_times': played_move_times,
            'player_at_pos': batch_metadata['player_at_idx']
            
        }
        
all_games = sorted(PLAYED_GAME_DATA.keys())

In [ ]:
human_watch_data = {game: {} for game in all_games}
human_watch_payoffs = {game: [] for game in all_games}
human_watch_fun = {game: [] for game in all_games}

for user, user_data in watch_data.items():
    for game, game_data in user_data.items(): 
        arena = game_data['batch_metadata']['arena']
        
        if arena not in human_watch_data[game]: 
            human_watch_data[game][arena] = {game_stage: {
                'dist': [], 'rt': [], 'uid': []
                } for game_stage in GAME_STAGES}
            
        payoff = game_data['judgements'][-1]
        if payoff is not None: 
            human_watch_payoffs[game].append(payoff)
        else: 
            if 'response' in game_data['judgements'][0]: human_watch_fun[game].append(game_data['judgements'][0]['response'])
            
        for order, entry in enumerate(game_data['pred_moves']):
            raw_pred_moves, _, _, _, rt = entry
            game_stage = GAME_STAGES[order]
            
            # convert click count format of keys: 'row-col' --> (row, col)
            pred_move_dist = {}
            n_rows_game, n_cols_game, win_conds = game.split("*")
            n_rows_game = int(float(n_rows_game))
            n_cols_game = int(float(n_cols_game))
            
            for r in range(n_rows_game): 
                for c in range(n_cols_game): 
                    pred_move_dist[(r, c)] = 0
            norm_counts = int(np.sum(list(raw_pred_moves.values())))
            for k, v in raw_pred_moves.items(): 
                row, col = k.split('-')
                row = int(row)
                col = int(col)
                k = (row,col)
                pred_move_dist[k] = v / norm_counts

            human_watch_data[game][arena][game_stage]['dist'].append(pred_move_dist)
            human_watch_data[game][arena][game_stage]['rt'].append(rt)
            human_watch_data[game][arena][game_stage]['uid'].append(user)

In [ ]:
agents = ['ours',
          'expert',
          'random']

model_watch_data = {agent: 
    {game: {} for game in PLAYED_GAME_DATA.keys()}
    for agent in agents
    }

for agent in agents: 
    for game in PLAYED_GAME_DATA.keys(): 
        game_id = game2idx[game]
        game_obj = games[game_id - 1]
        
        for arena, arena_data in PLAYED_GAME_DATA[game].items(): 
            model_watch_data[agent][game][arena] = {game_stage: [] for game_stage in GAME_STAGES}
            
            boards = arena_data['boards_at_stage']
            
            move_idxs = arena_data['move_idxs']
            
            for order, game_stage in enumerate(GAME_STAGES): 
                
                move_idx = move_idxs[order] + move_idx_offset
                current_board = boards[game_stage]
                # just take the plays 
                current_board = [[piece for piece, move_turn in entries] for entries in current_board]
                current_board = analysis_utils.convert_board_repr(current_board)
                    
                if agent != "random": 
                    raw_m_outputs = model_fit_data[agent][arena][str(game_obj['index'])][str(move_idx)]['dist']
                    if type(raw_m_outputs) != list: raw_m_outputs =[raw_m_outputs]
                    
                    # convert any -inf to 0 --- check?? or drop key?
                    m_outputs = []
                    for m_output in raw_m_outputs: 
                        new_dict = {}
                        for k, v in m_output.items():
                            if v != -np.inf: new_dict[k] = v
                        m_outputs.append(new_dict)
                    
                    
                    m_dist = [utils.softmax_dist(m_output, T=1) for m_output in m_outputs]
                else: 
                    
                    # get all possible legal moves, and use this to ground comparisons
                    m_dist = [uniform_dist(current_board)]
                    
                model_watch_data[agent][game][arena][game_stage] = m_dist

In [ ]:
stage_to_title = {
    "early": "Early",
    "intermediate": "Mid",
    "late": "Late"
}

stages = ['early', 'intermediate', 'late']

def plot_watch_data(game, arena, stages, viz_agents, fig=None, game_index=None, title=None, show_colorbar=True):

    if fig is None:
        fig, axes = plt.subplots(1 + len(viz_agents), len(stages), figsize=(8, 6))
    else:
        axes = fig.subplots(1 + len(viz_agents), len(stages))

    arena_data = PLAYED_GAME_DATA[game][arena]
    for order, game_stage in enumerate(GAME_STAGES):

        # Extract the player who moved and where they moved
        played_move = tuple(arena_data['played_pos'][order])
        player_index = arena_data['player_at_pos'][order]
        to_move = 'player_1' if player_index == 1 else 'player_2'
        move_sequence = arena_data['boards_at_stage'][game_stage]

        # Instantiate the current and subsequent boards
        current_board = np.array([[piece for piece, move_turn in entries] for entries in move_sequence])
        height, width = current_board.shape
        next_board = np.copy(current_board)
        next_board[played_move] = player_index
        cmap = matplotlib.colors.LinearSegmentedColormap.from_list("", [COLORS["empty"], COLORS["target"]])

        # Plot the watch distribution
        watch_distributions = human_watch_data[game][arena][game_stage]['dist']
        aggregated = analysis_utils.average_dicts(watch_distributions)
        dist = dist_from_dict(aggregated, height, width)
        im = render_dist(axes[0, order], distribution=dist, board=current_board, cmap=cmap, played_move=played_move, player_to_move=to_move)
        axes[0, 0].set_ylabel("Watch", fontweight="bold")

        # Plot the agents
        for j, agent in enumerate(viz_agents):
            model_dsitributions = model_watch_data[agent][game][arena][game_stage]
            aggregated = analysis_utils.average_dicts(model_dsitributions)
            dist = dist_from_dict(aggregated, height, width)
            render_dist(axes[j+1, order], distribution=dist, board=current_board, cmap=cmap, played_move=played_move, player_to_move=to_move)
            axes[j+1, 0].set_ylabel(AGENT_TO_TITLE[agent], fontweight="bold")

        # Set the x labels
        axes[-1, order].set_xlabel(stage_to_title[game_stage], fontweight="bold")

        # Set the colorbar for each column (i.e. game stage)
        if show_colorbar:
            cbar = fig.colorbar(im, ax=axes[:, order].ravel().tolist(), shrink=0.75)
            # cbar.ax.tick_params(size=0)

    # Set the overall figure title
    # Set the overall figure title
    if title is not None:
        formatted_title = title
    elif game_index is None:
        tidy_name = analysis_utils.tidy_game_code(game)
        fmt_name = "$\\bf{Game:}$ " + tidy_name
        fmt_arena = "$\\bf{Arena:}$ " + arena
        formatted_title = f"{fmt_name}\n{fmt_arena}"
    else:
        formatted_title = f"Game Index: {game_index}\nArena: {arena}"
    fig.suptitle(formatted_title, fontsize=10)

    if fig is None:
        plt.show()

    return fig, axes


### Saving each row separately

In [ ]:
POSITIVE_WATCH_EXAMPLES = [
    ("5x5 4 (P2 2p)", "01JNGYBT1XEF0P2DNVYKGTR57G", "intermediate", "5 by 5 board\n4 in a row wins\nP2 opens twice"),
    ("4x9 4", "01JNM7DPV5CPV91M934S8PSK3Z", "intermediate", "4 by 9 board\n4 in a row wins"),
    ("7x7 4 P1 / 3 P2", "01JP09AZZ6XV59PWN7A5C1CMZR", "early", "7 by 7 board\n4 in a row wins for P1\n3 in row wins for P2"),
    ("5x5 4 P1 / 3 P2",  "01JNGTBNEY2WJK72RKDXWA6283", "late", "5 by 5 board\n4 in a row wins for P1\n3 in row wins for P2"),
    ("1x10 3", "01JNF3TJETGCNRSJZPPGC4PFW7", "early", "1 by 10 board\n3 in a row wins"),
    ("10x10 4", "01JP10JWF19E77F0A2B6DTJJVK", "intermediate", "10 by 10 board\n4 in a row wins"),
]

NEGATIVE_WATCH_EXAMPLES = [
    ("7x7 4", "01JNHMYCXADS62X5A47WFYB180", "late", "7 by 7 board\n4 in a row wins"),
    ("4x4 3 HV", "01JP10J988720SWM7XJ5V22P7D", "late", "4 by 4 board\n3 in a row wins\nNo diagonals")
]

DOUBLE_FAILURE_WATCH_EXAMPLES = [
    ("5x5 4 (P1 2p)", "01JNEZFJ8J6WTS95ZT24GVW0XD", "late", "5 by 5 board\n4 in a row wins\nP1 opens twice"),
    ("4x4 3 L", "01JNH0JREQ0B4M5NYGS95FV36D", "intermediate", "4 by 4 board\n3 in a row loses")
]

In [ ]:
def prettify_name(game_name):
    board, other = game_name.split(" ")[0], " ".join(game_name.split(" ")[1:])

    row, width = board.split("x")
    pretty_board = f"{row} by {width} board"

    # Case 1: simple k in a row
    if " " not in other:
        pretty_remainder = f"{other} in a row wins"

    # Case 2: k in a row loses
    elif "L" in other:
        pretty_remainder = f"{other.replace('L', '').strip()} in a row loses"

    # Case 3: someone opens twice
    elif "2p" in other:
        k = other.split(" ")[0]
        if "P1" in other:
            pretty_remainder = f"{k} in a row wins\nP1 opens twice"
        elif "P2" in other:
            pretty_remainder = f"{k} in a row wins\nP2 opens twice"

    # Case 4: no diagonals
    elif "HV" in other:
        k = other.split(" ")[0]
        if "P1" in other:
            pretty_remainder = f"{k} in a row wins\nP1 can't win with diagonals"
        elif "P2" in other:
            pretty_remainder = f"{k} in a row wins\nP2 can't win with diagonals"
        else:
            pretty_remainder = f"{other.replace('HV', '').strip()} in a row wins\nPlayers can't win with diagonals"

    # Case 5: only diagonals
    elif "D" in other:
        k = other.split(" ")[0]
        if "P1" in other:
            pretty_remainder = f"{k} in a row wins\nP1 can only win with diagonals"
        elif "P2" in other:
            pretty_remainder = f"{k} in a row wins\nP2 can only win with diagonals"
        else:
            pretty_remainder = f"{other.replace('D', '').strip()} in a row wins\nPlayers can only win with diagonals"

    # Case 6: different win conditions for P1 and P2
    elif "/" in other:
        p1_info, p2_info = other.split(" / ")
        p1_k = p1_info.split(" ")[0]
        p2_k = p2_info.split(" ")[0]
        pretty_remainder = f"{p1_k} in a row wins for P1\n{p2_k} in a row wins for P2"

    else:
        pretty_remainder = "ERROR ERROR"

    return f"{pretty_board}\n{pretty_remainder}"

In [ ]:
ALL_GAMES = list(human_watch_data.keys())
MAX_NUM_ARENAS = 4
SEED = 42

os.makedirs("./final_figures/watch_data_by_row", exist_ok=True)

random.seed(SEED)

figure = plt.figure(figsize=(6 * MAX_NUM_ARENAS, 6 * len(ALL_GAMES)))
subfigs = figure.subfigures(len(ALL_GAMES), MAX_NUM_ARENAS)

for i, game in enumerate(ALL_GAMES):
    figure = plt.figure(figsize=(6 * MAX_NUM_ARENAS, 6))
    subfigs = figure.subfigures(1, MAX_NUM_ARENAS)

    all_arenas = list(human_watch_data[game].keys())

    for j, arena in enumerate(all_arenas):
        arena_index = human_gameplay_data[game]['arena'].index(arena)
        fig, axes = plot_watch_data(game, arena, GAME_STAGES, viz_agents+['random'], subfigs[j], show_colorbar=False, title="")

    tidy_name = analysis_utils.tidy_game_code(game)
    figure.suptitle("$\\bf{Game:}$ " + prettify_name(tidy_name), fontsize=24, y=1.08)
    figure.savefig(f"./final_figures/watch_data_by_row/{tidy_name.replace(' ', '_').replace('/', '&')}.pdf",
                    bbox_inches="tight")
    

    print("\nOriginal name:", tidy_name)
    print("Prettified name:", prettify_name(tidy_name))

### Watch

In [ ]:
SCALE_HIGHLIGHT = True

def plot_watch_single_stage(game, arena, game_stage, viz_agents, fig=None, game_index=None,
                            title=None, plot_played=True, show_colorbar=True, colorbar_anchor=(0, 0.5),
                            colorbar_location="right", colorbar_pad=0.05, colorbar_label="",
                            title_location='center'):
    
    n_top_plots = 2 if plot_played else 1

    if fig is None:
        fig, axes = plt.subplots(n_top_plots + len(viz_agents), 1, figsize=(8, 6))
    else:
        axes = fig.subplots(n_top_plots + len(viz_agents), 1)

    arena_data = PLAYED_GAME_DATA[game][arena]
    order = stages.index(game_stage)

    # Extract the player who moved and where they moved
    played_move = tuple(arena_data['played_pos'][order])
    player_index = arena_data['player_at_pos'][order]
    to_move = 'player_1' if player_index == 1 else 'player_2'
    move_sequence = arena_data['boards_at_stage'][game_stage]

    # Instantiate the current and subsequent boards
    current_board = np.array([[piece for piece, move_turn in entries] for entries in move_sequence])
    height, width = current_board.shape
    next_board = np.copy(current_board)
    next_board[played_move] = player_index

    # cmap = matplotlib.colors.LinearSegmentedColormap.from_list("", [COLORS["empty"], COLORS[to_move]])
    cmap = matplotlib.colors.LinearSegmentedColormap.from_list("", [COLORS["empty"], COLORS["target"]])

    if plot_played:
        render_dist(axes[0], distribution=np.zeros_like(current_board), board=next_board, cmap=cmap, highlight=played_move,
                    scale_highlight=SCALE_HIGHLIGHT)
        axes[0].set_ylabel("Played", fontweight="bold")
        watch_idx = 1
    else:
        watch_idx = 0

    # Plot the watch distribution
    watch_distributions = human_watch_data[game][arena][game_stage]['dist']
    aggregated = analysis_utils.average_dicts(watch_distributions)
    dist = dist_from_dict(aggregated, height, width)
    im = render_dist(axes[watch_idx], distribution=dist, board=current_board, cmap=cmap, played_move=played_move, player_to_move=to_move,
                     scale_highlight=SCALE_HIGHLIGHT)
    axes[watch_idx].set_ylabel("Human", fontweight="bold")
    watch_idx += 1

    # Plot the agents
    for j, agent in enumerate(viz_agents):
        model_dsitributions = model_watch_data[agent][game][arena][game_stage]
        aggregated = analysis_utils.average_dicts(model_dsitributions)
        dist = dist_from_dict(aggregated, height, width)
        render_dist(axes[j+watch_idx], distribution=dist, board=current_board, cmap=cmap, played_move=played_move, player_to_move=to_move,
                     scale_highlight=SCALE_HIGHLIGHT)
        axes[j+watch_idx].set_ylabel(AGENT_TO_TITLE[agent], fontweight="bold")

    # Set the colorbar
    if show_colorbar:
        cbar = fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.75, anchor=colorbar_anchor,
                            location=colorbar_location, pad=colorbar_pad)
        # cbar.ax.tick_params(size=0)
        if colorbar_label != "":
            cbar.ax.set_title(colorbar_label, fontsize=10, y=1.05)

    # Set the overall figure title
    if title is not None:
        formatted_title = title
    elif game_index is None:
        tidy_name = analysis_utils.tidy_game_code(game)
        fmt_name = "$\\bf{Game:}$ " + tidy_name
        fmt_arena = "$\\bf{Arena:}$ " + arena
        formatted_title = f"{fmt_name}\n{fmt_arena}"
    else:
        formatted_title = f"Game Index: {game_index}\nArena: {arena}"
    fig.suptitle(formatted_title, fontsize=10, y=1.05, ha=title_location)

    if fig is None:
        plt.show()

    return fig, axes
    

In [ ]:
POSITIVE_WATCH_EXAMPLES = [
    ("5x5 4 (P2 2p)", "01JNGYBT1XEF0P2DNVYKGTR57G", "intermediate", "5 by 5 board\n4 in a row wins\nP2 opens twice"),
    ("4x9 4", "01JNM7DPV5CPV91M934S8PSK3Z", "intermediate", "4 by 9 board\n4 in a row wins"),
    ("7x7 4 P1 / 3 P2", "01JP09AZZ6XV59PWN7A5C1CMZR", "early", "7 by 7 board\n4 in a row wins for P1\n3 in row wins for P2"),
    ("5x5 4 P1 / 3 P2",  "01JNGTBNEY2WJK72RKDXWA6283", "late", "5 by 5 board\n4 in a row wins for P1\n3 in row wins for P2"),
    ("1x10 3", "01JNF3TJETGCNRSJZPPGC4PFW7", "early", "1 by 10 board\n3 in a row wins"),
    ("10x10 4", "01JP10JWF19E77F0A2B6DTJJVK", "intermediate", "10 by 10 board\n4 in a row wins"),
]

NEGATIVE_WATCH_EXAMPLES = [
    ("7x7 4", "01JNHMYCXADS62X5A47WFYB180", "late", "7 by 7 board\n4 in a row wins"),
    ("4x4 3 HV", "01JP10J988720SWM7XJ5V22P7D", "late", "4 by 4 board\n3 in a row wins\nNo diagonals")
]

DOUBLE_FAILURE_WATCH_EXAMPLES = [
    ("5x5 4 (P1 2p)", "01JNEZFJ8J6WTS95ZT24GVW0XD", "late", "5 by 5 board\n4 in a row wins\nP1 opens twice"),
    ("4x4 3 L", "01JNH0JREQ0B4M5NYGS95FV36D", "intermediate", "4 by 4 board\n3 in a row loses")
]

ALL_WATCH_EXAMPLES = POSITIVE_WATCH_EXAMPLES + NEGATIVE_WATCH_EXAMPLES + DOUBLE_FAILURE_WATCH_EXAMPLES

In [ ]:
# Get the reverse mapping from tidy to full game title
all_play_games = list(human_gameplay_data.keys())

TIDY_TO_FULL = {}
for i, game in enumerate(all_play_games):
    tidy_name = analysis_utils.tidy_game_code(game)
    TIDY_TO_FULL[tidy_name] = game


In [ ]:
figure = plt.figure(figsize=(12, 8))
subfigs = figure.subfigures(2, 5)
watch_viz_agents = ['ours', 'expert']
letters = "abcdefghij"


for i, (tidy, arena, stage, manual_title) in enumerate(ALL_WATCH_EXAMPLES):
    row, col = i//5, i%5

    game = TIDY_TO_FULL[tidy]
    letter = letters[i]
    title = "$\\bf{" + letter + "}$   " + manual_title

    # Only show the colorbar in the bottom-right plot
    is_last = (i == len(ALL_WATCH_EXAMPLES) - 1)

    fig, axes = plot_watch_single_stage(game, arena, stage, watch_viz_agents, subfigs[row, col], title=title,
                                        plot_played=False, show_colorbar=is_last, colorbar_pad=0.35,
                                        colorbar_label="P(move)" if is_last else "",
                                        title_location='right' if is_last else 'center')

    for ax in axes:
        ax.set_anchor('C')

plt.show()
figure.savefig("./final_figures/selected_watch_examples.pdf", bbox_inches='tight')
figure.savefig("./final_figures/figure6_updated.pdf", bbox_inches='tight')